# Exp B — Diversity Metric Screening

**Hypothesis:** MI variance (CV of cluster-level MI) best predicts AISO success across datasets.

Three candidate metrics per dataset:
1. `k` — number of natural feature clusters
2. `CV(μ)` — coefficient of variation of cluster-level MI
3. `eff_rank` — effective rank of feature correlation matrix

Known AISO outcomes (SC PR-AUC delta vs random baseline, from paper):
| Dataset | Delta SC vs S0 | Verdict |
|---------|---------------|----------|
| Elliptic | +0.111 | success |
| Amazon | +0.020 (SC < SB) | partial fail |
| YelpChi | -0.014 | fail |
| CICIDS2017 | negative | fail |

**Kernel:** .venv (sklearn only)

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import spearmanr
import warnings; warnings.filterwarnings('ignore')

# Known AISO outcomes from paper (SC PR-AUC - S0 PR-AUC)
OUTCOMES = {
    'Elliptic':   +0.111,   # 0.6644 - 0.5530
    'Amazon':     +0.020,   # 0.4940 - 0.4743 (SC < SB → partial fail)
    'YelpChi':    -0.014,   # 0.2249 - 0.2388
    'CICIDS2017': -0.050,   # approximate, Smart M -14% vs mRMR
}

print('Outcomes loaded:', OUTCOMES)

Outcomes loaded: {'Elliptic': 0.111, 'Amazon': 0.02, 'YelpChi': -0.014, 'CICIDS2017': -0.05}


## Metric computation function

In [10]:
def compute_diversity_metrics(X, y, k_range=range(5, 25), seed=42):
    """Returns dict of diversity metrics for a dataset."""
    N, D = X.shape
    
    # MI per feature
    MI = mutual_info_classif(X, y, random_state=seed)
    
    # Correlation matrix
    C_abs = np.abs(np.corrcoef(X.T))
    np.fill_diagonal(C_abs, 0.0)
    dist_mx = 1.0 - C_abs; np.fill_diagonal(dist_mx, 0.0)
    
    # Find natural k via gap in within-cluster MI variance
    best_k, best_cv = 10, 0
    for k in k_range:
        if k >= D: continue
        cl = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
        labels = cl.fit_predict(dist_mx)
        mu_c = np.array([MI[labels == c].mean() for c in range(k)])
        cv = mu_c.std() / (mu_c.mean() + 1e-8)
        if cv > best_cv:
            best_cv = cv
            best_k  = k
    
    # Compute metrics at best_k
    cl = AgglomerativeClustering(n_clusters=best_k, metric='precomputed', linkage='average')
    labels = cl.fit_predict(dist_mx)
    mu_c   = np.array([MI[labels == c].mean() for c in range(best_k)])
    
    # 1. k
    metric_k = best_k
    
    # 2. CV(μ) = std/mean of cluster MI
    metric_cv = mu_c.std() / (mu_c.mean() + 1e-8)
    
    # 3. Effective rank = exp(entropy of normalized singular values)
    C_full = np.corrcoef(X.T)
    svs = np.linalg.svd(C_full, compute_uv=False)
    svs = np.abs(svs); svs /= svs.sum() + 1e-8
    eff_rank = np.exp(-np.sum(svs * np.log(svs + 1e-12)))
    
    # 4. Cross-cluster mean correlation
    cross_corr = []
    for i in range(best_k):
        for j in range(i+1, best_k):
            fi = np.where(labels == i)[0]; fj = np.where(labels == j)[0]
            cross_corr.append(np.mean(C_abs[np.ix_(fi, fj)]))
    metric_cross = np.mean(cross_corr) if cross_corr else 0.0
    
    return {
        'k': metric_k,
        'CV_MI': metric_cv,
        'eff_rank': eff_rank,
        'cross_corr': metric_cross,
        'MI_max': MI.max(),
        'MI_mean': MI.mean(),
    }

print('compute_diversity_metrics ready')

compute_diversity_metrics ready


## Elliptic Bitcoin

In [11]:
DATA = Path('../Elliptic Bitcoin/elliptic_bitcoin_dataset')
feat_df = pd.read_csv(DATA / 'elliptic_txs_features.csv', header=None)
cls_df  = pd.read_csv(DATA / 'elliptic_txs_classes.csv')
N_FEAT  = feat_df.shape[1] - 2
feat_df.columns = ['txId','timestep'] + [f'f{i}' for i in range(N_FEAT)]
cls_df.columns  = ['txId','class']
df = feat_df.merge(cls_df, on='txId')
df = df[df['class'] != 'unknown'].copy()
df['label'] = (df['class'] == '1').astype(int)
train = df[df['timestep'] <= 34]
X_e = train[[f'f{i}' for i in range(N_FEAT)]].values.astype(float)
y_e = train['label'].values

print('Computing Elliptic metrics...')
metrics_elliptic = compute_diversity_metrics(X_e, y_e)
print(metrics_elliptic)

Computing Elliptic metrics...
{'k': 12, 'CV_MI': np.float64(0.667396416634905), 'eff_rank': np.float64(47.15045590608216), 'cross_corr': np.float64(0.018258094913384364), 'MI_max': np.float64(0.23027935267919286), 'MI_mean': np.float64(0.07972656035420242)}


## Amazon
> Adjust path if needed

In [12]:
import scipy.io
mat = scipy.io.loadmat(Path('../Amazon/Amazon.mat'))
X_a = mat['features'].toarray().astype(float)
y_a = mat['label'].flatten().astype(int)
print(f'Amazon: X={X_a.shape}, y={y_a.shape}, fraud={y_a.sum()} ({y_a.mean()*100:.1f}%)')

print('Computing Amazon metrics...')
metrics_amazon = compute_diversity_metrics(X_a, y_a)
print(metrics_amazon)

Amazon: X=(11944, 25), y=(11944,), fraud=821 (6.9%)
Computing Amazon metrics...
{'k': 9, 'CV_MI': np.float64(1.6011026905093493), 'eff_rank': np.float64(9.760218946542821), 'cross_corr': np.float64(0.09200425338840817), 'MI_max': np.float64(0.16414799908380595), 'MI_mean': np.float64(0.03478503651089863)}


## YelpChi

In [13]:
mat2 = scipy.io.loadmat(Path('../YelpChi/YelpChi.mat'))
X_y = mat2['features'].toarray().astype(float)
y_y = mat2['label'].flatten().astype(int)
print(f'YelpChi: X={X_y.shape}, y={y_y.shape}, fraud={y_y.sum()} ({y_y.mean()*100:.1f}%)')

print('Computing YelpChi metrics...')
metrics_yelpchi = compute_diversity_metrics(X_y, y_y)
print(metrics_yelpchi)

YelpChi: X=(45954, 32), y=(45954,), fraud=6677 (14.5%)
Computing YelpChi metrics...
{'k': 23, 'CV_MI': np.float64(1.732887655534835), 'eff_rank': np.float64(21.121314527879434), 'cross_corr': np.float64(0.054888564427795564), 'MI_max': np.float64(0.20306982885923142), 'MI_mean': np.float64(0.025613860542492387)}


## CICIDS2017

In [ ]:
metrics_cicids = None  # CICIDS skipped — not required per draft paper
print('CICIDS: skipped')

In [ ]:
# Assemble results
all_metrics = {
    'Elliptic':   metrics_elliptic,
    'Amazon':     metrics_amazon,
    'YelpChi':    metrics_yelpchi,
    'CICIDS2017': metrics_cicids,
}

available = {k: v for k, v in all_metrics.items() if v is not None}
datasets  = list(available.keys())
outcomes  = [OUTCOMES[d] for d in datasets]

if len(available) < 2:
    print('Need at least 2 datasets to correlate — load more datasets above')
else:
    metric_names = ['k', 'CV_MI', 'eff_rank', 'cross_corr']
    print(f'{"Metric":<15} {"Spearman r":>12} {"p-value":>10}')
    print('-' * 40)
    for m in metric_names:
        vals = [available[d][m] for d in datasets]
        r, p = spearmanr(vals, outcomes)
        print(f'  {m:<13} {r:>12.3f} {p:>10.3f}')
    
    print('\nHypothesis: CV_MI has highest |r|')
    
    # Table
    print(f'\n{"Dataset":<12} {"k":>4} {"CV_MI":>8} {"eff_rank":>10} {"Delta SC":>10}')
    print('-' * 50)
    for d in datasets:
        m = available[d]
        print(f'  {d:<10} {m["k"]:>4} {m["CV_MI"]:>8.3f} {m["eff_rank"]:>10.1f} {OUTCOMES[d]:>10.3f}')

Metric            Spearman r    p-value
----------------------------------------
  k                    1.000        nan
  CV_MI               -1.000        nan
  eff_rank             1.000        nan
  cross_corr          -1.000        nan

Hypothesis: CV_MI has highest |r|

Dataset         k    CV_MI   eff_rank   Delta SC
--------------------------------------------------
  Elliptic     12    0.667       47.2      0.111
  Amazon        9    1.601        9.8      0.020
